In [1]:
%cd '/home/mojtaba.kananisarche/'

/home/mojtaba.kananisarche


In [10]:
import numpy as np
import keras

class MyDataGenerator(keras.utils.Sequence):
    """
    Custom data generator for Keras.
    """

    def __init__(self, list_IDs, labels, batch_size=32, shuffle=True):
        """
        Initialization function.  This is where you set up the basic parameters
        of your data generator.
        """
        self.batch_size = batch_size
        self.labels = labels
        self.list_IDs = list_IDs
        self.shuffle = shuffle
        self.on_epoch_end()

    def __len__(self):
        """
        Returns the number of batches per epoch.  Keras uses this to know how many
        times to call `__getitem__` in one epoch.
        """
        pass  # Replace with your code

    def __getitem__(self, index):
        """
        Generates one batch of data.  The core of the data generator.
        """
        pass  # Replace with your code

    def on_epoch_end(self):
        """
        Called at the end of each epoch.  Use this to re-shuffle the data if needed.
        """
        pass  # Replace with your code

    def __data_generation(self, list_IDs_temp):
        """
        Generates data containing batch_size samples. This is where the actual data
        loading and preprocessing happens.
        """
        pass # Replace with your code


2025-04-21 03:18:17.753285: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [110]:
from typing import Optional, List, Dict
import pandas as pd
%cd '/home/mojtaba.kananisarche/'
from Zh.src.utils.file_folder import get_file_paths
import pandas as pd
import os

def _get_train_val_test_file_paths(
    input_path: str,
    primary_column_name: str,
    secondary_column_name: Optional[str] = None,
) -> List[Dict]:

    file_paths = get_file_paths(
        folder_path = input_path,
        depth = 2,
        EXT_LIST = ['.csv']
    )
    data = pd.DataFrame()
    data['path'] = file_paths
    data['fold'] = data['path'].apply(lambda x: os.path.basename(x).split('_')[0]).astype(int).astype(str)
    data['subject_id'] = data['path'].apply(lambda x: os.path.basename(x).split('_')[1]).astype(int).astype(str)
    data = data.sort_values(['fold', 'subject_id']).reset_index(drop=True)

    columns_to_fold_by = data[[primary_column_name]+ ([secondary_column_name] if secondary_column_name else [])]

    train_val_test_file_paths = []
    unique_main_column_values = sorted(
        columns_to_fold_by[primary_column_name].unique()
    )

    if primary_column_name and secondary_column_name:
        for main_column_value in unique_main_column_values:
            corresponding_secondary_value = columns_to_fold_by[
                columns_to_fold_by[primary_column_name] == main_column_value
            ][secondary_column_name].unique()
            assert (
                len(corresponding_secondary_value) == 1
            ), f"Multiple Unique Secondary Values Found For Main Column Value {main_column_value} - {corresponding_secondary_value}"
            corresponding_secondary_value = corresponding_secondary_value[0]

            fold_train_val_test_file_paths = {"info": {}, "indices": {}}

            test_split = columns_to_fold_by[
                columns_to_fold_by[primary_column_name] == main_column_value
            ]
            fold_train_val_test_file_paths["indices"][
                "test"
            ] = data.loc[test_split.index.to_list()]['path'].to_list()
            fold_train_val_test_file_paths["info"]["test"] = {
                f"{primary_column_name}": sorted(
                    test_split[primary_column_name].unique().tolist()
                )
            }

            val_split = columns_to_fold_by[
                (columns_to_fold_by[primary_column_name] != main_column_value)
                & (
                    columns_to_fold_by[secondary_column_name]
                    == corresponding_secondary_value
                )
            ]
            fold_train_val_test_file_paths["indices"][
                "val"
            ] = data.loc[val_split.index.to_list()]['path'].to_list()
            fold_train_val_test_file_paths["info"]["val"] = {
                f"{primary_column_name}": sorted(
                    val_split[primary_column_name].unique().tolist()
                )
            }

            train_split = columns_to_fold_by[
                columns_to_fold_by[secondary_column_name]
                != corresponding_secondary_value
            ]
            fold_train_val_test_file_paths["indices"][
                "train"
            ] = data.loc[train_split.index.to_list()]['path'].to_list()
            fold_train_val_test_file_paths["info"]["train"] = {
                f"{primary_column_name}": sorted(
                    train_split[primary_column_name].unique().tolist()
                )
            }
            train_val_test_file_paths.append(fold_train_val_test_file_paths)

    elif primary_column_name:
        for main_column_value_i in unique_main_column_values:
            for main_column_value_j in unique_main_column_values:
                if main_column_value_i != main_column_value_j:
                    fold_train_val_test_file_paths = {"info": {}, "indices": {}}

                    test_split = columns_to_fold_by[
                        columns_to_fold_by[primary_column_name]
                        == main_column_value_i
                    ]
                    fold_train_val_test_file_paths["indices"][
                        "test"
                    ] = data.loc[test_split.index.to_list()]['path'].to_list()
                    fold_train_val_test_file_paths["info"]["test"] = {
                        f"{primary_column_name}": sorted(
                            test_split[primary_column_name].unique().tolist()
                        )
                    }

                    val_split = columns_to_fold_by[
                        columns_to_fold_by[primary_column_name]
                        == main_column_value_j
                    ]
                    fold_train_val_test_file_paths["indices"][
                        "val"
                    ] = data.loc[val_split.index.to_list()]['path'].to_list()
                    fold_train_val_test_file_paths["info"]["val"] = {
                        f"{primary_column_name}": sorted(
                            val_split[primary_column_name].unique().tolist()
                        )
                    }

                    train_split = columns_to_fold_by[
                        ~columns_to_fold_by[primary_column_name].isin(
                            [main_column_value_i, main_column_value_j]
                        )
                    ]
                    fold_train_val_test_file_paths["indices"][
                        "train"
                    ] = data.loc[train_split.index.to_list()]['path'].to_list()
                    fold_train_val_test_file_paths["info"]["train"] = {
                        f"{primary_column_name}": sorted(
                            train_split[primary_column_name].unique().tolist()
                        )
                    }

                    train_val_test_file_paths.append(fold_train_val_test_file_paths)

    return train_val_test_file_paths


/home/mojtaba.kananisarche


In [111]:
input_path = "/work/long_lab/Mojtaba/data/UTA/1.3.2-augmented_embedded_preprocessed"
primary_column_name = 'fold'
secondary_column_name = None
train_val_test_file_paths = _get_train_val_test_file_paths(
    input_path = input_path, 
    primary_column_name = primary_column_name, 
    secondary_column_name = secondary_column_name, 
)

In [112]:
train_val_test_file_paths

[{'info': {'test': {'fold': ['1']},
   'val': {'fold': ['2']},
   'train': {'fold': ['3', '4', '5']}},
  'indices': {'test': ['/work/long_lab/Mojtaba/data/UTA/1.3.2-augmented_embedded_preprocessed/brightness_alpha0.8_beta-10/1_01_0_30fps_brightness_alpha0.8_beta-10_preprocessed.csv',
    '/work/long_lab/Mojtaba/data/UTA/1.3.2-augmented_embedded_preprocessed/brightness_alpha0.8_beta-10/1_01_5_30fps_brightness_alpha0.8_beta-10_preprocessed.csv',
    '/work/long_lab/Mojtaba/data/UTA/1.3.2-augmented_embedded_preprocessed/brightness_alpha0.8_beta-10/1_01_10_30fps_brightness_alpha0.8_beta-10_preprocessed.csv',
    '/work/long_lab/Mojtaba/data/UTA/1.3.2-augmented_embedded_preprocessed/contrast_alpha0.7_beta0/1_01_0_30fps_contrast_alpha0.7_beta0_preprocessed.csv',
    '/work/long_lab/Mojtaba/data/UTA/1.3.2-augmented_embedded_preprocessed/contrast_alpha0.7_beta0/1_01_5_30fps_contrast_alpha0.7_beta0_preprocessed.csv',
    '/work/long_lab/Mojtaba/data/UTA/1.3.2-augmented_embedded_preprocessed/con

In [92]:
train_val_test_indice[0]['indices']['val']

['/work/long_lab/Mojtaba/data/UTA/1.3.2-augmented_embedded_preprocessed/brightness_alpha0.8_beta-10/2_13_0_30fps_brightness_alpha0.8_beta-10_preprocessed.csv',
 '/work/long_lab/Mojtaba/data/UTA/1.3.2-augmented_embedded_preprocessed/contrast_alpha0.7_beta0/2_13_0_30fps_contrast_alpha0.7_beta0_preprocessed.csv',
 '/work/long_lab/Mojtaba/data/UTA/1.3.2-augmented_embedded_preprocessed/flip_flipcode1/2_13_0_30fps_flip_flipcode1_preprocessed.csv',
 '/work/long_lab/Mojtaba/data/UTA/1.3.2-augmented_embedded_preprocessed/rotate_angle-5/2_13_0_30fps_rotate_angle-5_preprocessed.csv',
 '/work/long_lab/Mojtaba/data/UTA/1.3.2-augmented_embedded_preprocessed/brightness_alpha1.2_beta10/2_13_0_30fps_brightness_alpha1.2_beta10_preprocessed.csv',
 '/work/long_lab/Mojtaba/data/UTA/1.3.2-augmented_embedded_preprocessed/grayscale/2_13_0_30fps_grayscale_preprocessed.csv',
 '/work/long_lab/Mojtaba/data/UTA/1.3.2-augmented_embedded_preprocessed/rotate_angle5/2_13_0_30fps_rotate_angle5_preprocessed.csv',
 '/wor

In [34]:
%cd '/home/mojtaba.kananisarche/'
from Zh.src.utils.file_folder import get_file_paths
import pandas as pd
import os

input_path = "/work/long_lab/Mojtaba/data/UTA/1.3.2-augmented_embedded_preprocessed"
file_paths = get_file_paths(
    folder_path = input_path,
    depth = 2,
    EXT_LIST = ['.csv']
)
data = pd.DataFrame()
data['path'] = file_paths
data['filename'] = data['path'].apply(lambda x: os.path.basename(x))
data['fold'] = data['path'].apply(lambda x: os.path.basename(x).split('_')[0]).astype(int).astype(str)
data['subject_id'] = data['path'].apply(lambda x: os.path.basename(x).split('_')[1]).astype(int).astype(str)
data['target'] = data['path'].apply(lambda x: os.path.basename(x).split('_')[2])
data = data.sort_values(['fold', 'subject_id', 'target']).reset_index(drop=True)

/home/mojtaba.kananisarche


In [35]:
data

,path,filename,fold,subject_id,target
0,/work/long_lab/Mojtaba/data/UTA/1.3.2-augmente...,1_01_0_30fps_brightness_alpha0.8_beta-10_prepr...,1,1,0
1,/work/long_lab/Mojtaba/data/UTA/1.3.2-augmente...,1_01_0_30fps_contrast_alpha0.7_beta0_preproces...,1,1,0
2,/work/long_lab/Mojtaba/data/UTA/1.3.2-augmente...,1_01_0_30fps_flip_flipcode1_preprocessed.csv,1,1,0
3,/work/long_lab/Mojtaba/data/UTA/1.3.2-augmente...,1_01_0_30fps_rotate_angle-5_preprocessed.csv,1,1,0
4,/work/long_lab/Mojtaba/data/UTA/1.3.2-augmente...,1_01_0_30fps_brightness_alpha1.2_beta10_prepro...,1,1,0
...,...,...,...,...,...
1570,/work/long_lab/Mojtaba/data/UTA/1.3.2-augmente...,5_60_5_30fps_brightness_alpha1.2_beta10_prepro...,5,60,5
1571,/work/long_lab/Mojtaba/data/UTA/1.3.2-augmente...,5_60_5_30fps_grayscale_preprocessed.csv,5,60,5
1572,/work/long_lab/Mojtaba/data/UTA/1.3.2-augmente...,5_60_5_30fps_rotate_angle5_preprocessed.csv,5,60,5
1573,/work/long_lab/Mojtaba/data/UTA/1.3.2-augmente...,5_60_5_30fps_cropped_preprocessed.csv,5,60,5


In [113]:
%cd '/home/mojtaba.kananisarche/'

from Zh.src.utils.load_data import load_data
import numpy as np
import pandas as pd
from typing import List, Optional, Tuple, Optional
from Zh.src.utils.file_folder import create_folders
import os
import json
import pandas as pd
import dataframe_image as dfi
from Zh.src.utils.visualize_results import confusion_matrix_results, table_results

/home/mojtaba.kananisarche


In [114]:
path = "/home/mojtaba.kananisarche/Zh/outputs/2025-04-20|09:08:09|RNN_FACENET_5CV"

In [115]:
data = table_results(
results_directory_path  = path,
output_directory_path   = None,
output_file_path_prefix = 'results',
cmap                    = "Blues",
reverse_cmap            = False,
header_1_bg_color       = 'silver',
header_1_text_color     = 'black',
header_2_bg_color       = 'darkgrey',
header_2_text_color     = 'black',
maximum_tries           = 1,
slice_or_video          = 'video', #slice, video
video_voting_method     = 'class_average', #majority, class_average, average
height                  = 2,
width                   = 5,
)

Try Number 0 Succeeded.


In [116]:
data = confusion_matrix_results(
    results_directory_path  = path,
    unique_classes          = [2, 6.5, 8.5],
    output_directory_path   = None,
    output_file_name_prefix = "results",
    slice_or_video          = 'slice',     #slice, video
    video_voting_method     = 'average',   #majority, class_average, average
    index                   = -1,          # positive for any index and negative for all
    split                   = "all",       # train, val, test, all
    cmap                    = "Greys",
    maximum_tries           = 5,
    corner_text_color       = "white",
    corner_bg_color         = "IndianRed",
    pred_true_text_color    = "black",
    pred_true_bg_color      = "silver",
    labels_bg_color         = "beige",
    labels_text_color       = "black",
    height                  = 5,
    width                   = None,
)

split - train
Try Number 0 Succeeded.
split - test
Try Number 0 Succeeded.
split - val
Try Number 0 Succeeded.
